# Web Scraping - Indeed.com
General steps for Web Scraping
1. Check whether the website allows web scraping
2. Obtain the source code (HTML File) by using the website URL
3. Download the website content
4. Parse the content using keywords tags for elements of interest
5. Extract relevant data/features
6. Organize raw data in structured format (e.g., CSV)

### Import Dependencies 

In [1]:
!pip install selenium

In [2]:
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
# from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service

### Path to webdriver (Firefox, Chrome) 

In [3]:
# Ensure that the driver path is correct before running this script.
# Microsoft Windows
driver_path = "D:\chromedriver\chromedriver.exe"
# Linux
#driver_path = "./drivers/linux/geckodriver"
service = Service(executable_path=driver_path)
driver = webdriver.Chrome(service=service)

NameError: name 'webdriver' is not defined

### Define position and location 

In [8]:
## Enter a job position
position = "data scientist"
## Enter a location (City, State or Zip or remote)
locations = "remote"

def get_url(position, location):
    url_template = "https://www.indeed.com/jobs?q={}&l={}"
    url = url_template.format(position, location)
    return url

url = get_url(position, locations)
dataframe = pd.DataFrame(columns=["Title", "Company", "Location", "Rating", "Date", "Salary", "Description", "Links"])

### Scrape job postings

In [9]:
## Number of postings to scrape
postings = 100

jn=0
for i in range(0, postings, 10):
    driver.get(url + "&start=" + str(i))
    driver.implicitly_wait(3)

    jobs = driver.find_elements(By.CLASS_NAME, 'job_seen_beacon')

    for job in jobs:
        result_html = job.get_attribute('innerHTML')
        soup = BeautifulSoup(result_html, 'html.parser')
        
        jn += 1
        
        liens = job.find_elements(By.TAG_NAME, "a")
        links = liens[0].get_attribute("href")
        
        title = soup.select('.jobTitle')[0].get_text().strip()
        company = soup.select('.companyName')[0].get_text().strip()
        location = soup.select('.companyLocation')[0].get_text().strip()
        try:
            salary = soup.select('.salary-snippet-container')[0].get_text().strip()
        except:
            salary = 'NaN'
        try:
            rating = soup.select('.ratingNumber')[0].get_text().strip()
        except:
            rating = 'NaN'
        try:
            date = soup.select('.date')[0].get_text().strip()
        except:
            date = 'NaN'
        try:
            description = soup.select('.job-snippet')[0].get_text().strip()
        except:
            description = ''
       
        dataframe = pd.concat([dataframe, pd.DataFrame([{'Title': title,
                                          "Company": company,
                                          'Location': location,
                                          'Rating': rating,
                                          'Date': date,
                                          "Salary": salary,
                                          "Description": description,
                                          "Links": links}])], ignore_index=True)
        print("Job number {0:4d} added - {1:s}".format(jn,title))

Job number    1 added - Data Scientist
Job number    2 added - Data Scientist
Job number    3 added - Jr. Data Scientist
Job number    4 added - Data Scientist - RWD
Job number    5 added - Data Scientist (Hybrid)
Job number    6 added - Senior Data Scientist
Job number    7 added - Data Scientist(R,Python,Linux,Biobank,NHS data,EHR-phenotyping)
Job number    8 added - Interdisciplinary-Microbiologist/Data Scientist
Job number    9 added - Data Scientist-Health
Job number   10 added - Data Scientist
Job number   11 added - Data Scientist
Job number   12 added - Data Scientist
Job number   13 added - Data Scientist
Job number   14 added - Data Scientist, Customer Experimentation
Job number   15 added - Jr. Data Scientist
Job number   16 added - Data Scientist, Commercial
Job number   17 added - Senior Data Scientist, Data Platform
Job number   18 added - Data Scientist
Job number   19 added - Data Scientist
Job number   20 added - Data Engineer/Data Scientist
Job number   21 added - Dat

In [10]:
driver.quit()

### Scrape full job descriptions

In [11]:
Links_list = dataframe['Links'].tolist()
#Links_list

In [12]:
import random
import time

In [13]:
driver = webdriver.Chrome(service=service)
descriptions=[]
for i in Links_list:
    driver.get(i)
    driver.implicitly_wait(random.randint(3, 8))
    jd = driver.find_element(By.XPATH, '//div[@id="jobDescriptionText"]').text
    descriptions.append(jd)
    time.sleep(random.randint(5,10))

dataframe['Descriptions'] = descriptions

In [14]:
driver.quit()

### Save results

In [15]:
# Convert the dataframe to a csv file
date = datetime.today().strftime('%Y-%m-%d')
dataframe.to_csv(date + "_" + position + "_" + locations + ".csv", index=False)

In [16]:
dataframe

,Title,Company,Location,Rating,Date,Salary,Description,Links,Descriptions
0,Data Scientist,Great American Insurance Company,"Remote in Los Angeles, CA 90017",3.8,PostedPosted 2 days ago,"$95,000 - $112,000 a year",Experience: 2+ years of experience with struct...,https://www.indeed.com/rc/clk?jk=7e518ef57fbbc...,Be Here. Be Great. Working for a leader in the...
1,Data Scientist,Tekwissenllc,"Remote in New York, NY 10112",NaN,PostedToday,$60 - $85 an hour,"Develop predictive models using statistical, m...",https://www.indeed.com/company/Tekwissenllc/jo...,Overview:\nTekWissen Group is a workforce mana...
2,Jr. Data Scientist,Net2Aspire,Remote,NaN,EmployerActive 1 day ago,"$65,000 - $80,000 a year", Create data dashboards and other data visual...,https://www.indeed.com/company/net2aspire/jobs..., Apply Statistical and Machine Learning metho...
3,Data Scientist - RWD,Norstella,Remote,NaN,EmployerActive 2 days ago,"$125,000 - $175,000 a year",Design data pipelines and queries and analyze ...,https://www.indeed.com/company/NorStella/jobs/...,Job Summary:\nWe are seeking an experienced Da...
4,Data Scientist (Hybrid),Baltimore Gas and Electric Company (BGE),Remote,NaN,PostedPosted 1 day ago,NaN,Apply the scientific method to extract knowled...,https://www.indeed.com/rc/clk?jk=a3f7b238a62e3...,"Description\nWe're powering a cleaner, brighte..."
...,...,...,...,...,...,...,...,...,...
145,Senior Data Scientist (Remote),Verikai,"Remote in San Francisco, CA",NaN,PostedPosted 30+ days ago,"$154,000 - $174,000 a year","With this data, we help insurance companies im...",https://www.indeed.com/rc/clk?jk=46791e6ebd142...,Company Introduction\nVerikai is an insurance ...
146,"Data Scientist or Statistician, Postdoc",UT Southwestern Medical Center,Remote,3.8,EmployerActive 2 days ago,"$67,000 - $120,000 a year","For data scientists, the projects include asse...",https://www.indeed.com/company/UT-Southwestern...,Center Information:\nThe Quantitative Biomedic...
147,Data Scientist,Flock Safety,"Remote in Atlanta, GA+1 location",2.7,PostedPosted 30+ days ago,NaN,Experience modeling data to drive analytics.\n...,https://www.indeed.com/rc/clk?jk=7d34cb7b5c324...,Who is Flock?\nFlock Safety provides the first...
148,Graduate - Data Science Analyst,Volkswagen Group of America - Chattanooga...,Remote,NaN,PostedToday,NaN,Explore and develop proof of concept (POC) sol...,https://www.indeed.com/rc/clk?jk=48de8bf0e9693...,Graduate - Data Science Analyst\n- ENT000011 -...
